# STS-B validation fine-threshold sweep

This notebook evaluates **only** the Hugging Face GLUE STS-B `validation` split. It binarizes human scores with `score >= 4`, embeds both sentences with `BAAI/bge-small-en-v1.5`, and evaluates cosine thresholds from 0.80 through 0.99 in 0.01 increments. Running the analysis saves the metrics CSV, error-analysis CSV, plots, and reproducibility metadata in `benchmarks/results/`. It does not load or evaluate the test split.

In [ ]:
from IPython.display import Image, display

from benchmarks.stsb_fine_sweep import run_stsb_validation_sweep

report = await run_stsb_validation_sweep()
print(f"Dataset: {report['dataset']} / {report['split']}")
print(f"Examples: {report['example_count']}; positives: {report['positive_count']}")
print(f"Test split used: {report['test_split_used']}")

In [ ]:
selected = report['selected_thresholds']
for summary in selected:
    print(
        f"{summary.name}: threshold={summary.threshold}, "
        f"precision={summary.precision}, recall={summary.recall}, "
        f"F1={summary.f1}, hits={summary.hit_count}, "
        f"false_positives={summary.false_positive_count}"
    )

for group in (
    'false_positive_at_0.92',
    'false_positive_at_0.96',
    'false_negative_at_0.88',
):
    examples = [
        row for row in report['error_analysis']
        if row['error_group'] == group
    ]
    print(f"\n{group}: {len(examples)} example(s)")
    for example in examples:
        print(
            f"sentence 1: {example['candidate_sentence']}\n"
            f"sentence 2: {example['query_sentence']}\n"
            f"cosine: {example['cosine_score']:.6f}; "
            f"gold score: {example['gold_score']:.2f}\n"
        )

In [ ]:
display(report['threshold_results'])
print('Run metadata:')
display(report['metadata'])
print('Saved files:')
display(report['files'])

In [ ]:
for key in ('metrics_plot', 'precision_recall_plot'):
    display(Image(filename=report['files'][key]))